# 실습 1 · Response-Based KD

같은 영상의 **Teacher와 Student 출력 확률**을 비교합니다. Teacher가 가장 유력한 클래스에 준 확률뿐 아니라 다른 후보에 얼마를 배정했는지도 Student의 학습 신호로 사용합니다.

실행 순서는 **Teacher 준비 → Temperature 비교 → Student 학습·평가**입니다.

## 1. 실행 환경과 CIFAR-10

처음부터 순서대로 실행합니다. CUDA가 있으면 GPU를 사용하며, CIFAR-10은 캐시가 없는 런타임에서 내려받습니다.

기본 실습은 **학습 64배치, 평가 512장**입니다. 난수 시드를 고정해 테스트 샘플을 선택하고, 학습 배치와 평가 데이터를 분리합니다.

**코드 셀 1 · 라이브러리와 실습 설정**

라이브러리를 불러오고 실행 장치, 데이터·가중치 경로, 무작위 시드, 학습 64배치와 테스트 512장 설정을 정합니다.

In [ ]:
# 실습 설정을 한곳에서 관리한다. 데이터·가중치 캐시를 재사용하고 CUDA가 있으면 GPU로 실행한다.
import copy
import os
import time
from itertools import islice
from pathlib import Path
from statistics import mean

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# 동일한 시드와 배치 수를 사용해 반복 실행 조건을 맞춘다.
SEED = 50
BATCH_SIZE = 64
TRAIN_STEPS = 64
TEST_SAMPLES = 512
DATA_ROOT = Path(os.environ.get("AX_IDEC_DATA_ROOT", "./data"))
ASSET_ROOT = Path(os.environ.get("AX_IDEC_CACHE", "./idec_cache"))
ASSET_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
print("실행 장치:", torch.cuda.get_device_name() if DEVICE.type == "cuda" else "CPU")

**코드 셀 2 · CIFAR-10 데이터 준비**

CIFAR-10의 학습·테스트 데이터를 내려받아 RGB 채널을 정규화하고, 고정된 샘플을 64장씩 전달하는 Loader를 만듭니다.

In [ ]:
# CIFAR-10 학습·테스트 분할에 같은 RGB 정규화를 적용한다. 테스트는 학습에 쓰이지 않는다.
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2023, 0.1994, 0.2010)
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])
train_data = datasets.CIFAR10(DATA_ROOT, train=True, download=True, transform=transform)
test_data = datasets.CIFAR10(DATA_ROOT, train=False, download=True, transform=transform)

# 시드를 고정해 일부 샘플만 선택하고, 같은 데이터에서 반복 비교할 Loader를 만든다.
def make_loader(dataset, samples, seed):
    indices = torch.randperm(len(dataset), generator=torch.Generator().manual_seed(seed))
    subset = Subset(dataset, indices[:samples].tolist())
    return DataLoader(subset, batch_size=BATCH_SIZE, num_workers=0,
                      pin_memory=(DEVICE.type == "cuda"))

train_loader = make_loader(train_data, (TRAIN_STEPS + 1) * BATCH_SIZE, SEED)
test_loader = make_loader(test_data, TEST_SAMPLES, SEED + 1)
print(f"학습 {TRAIN_STEPS} batches / 평가 {len(test_loader.dataset)} images")

## 2. 공통 Student 모델

경량 ResNet-18은 BasicBlock의 공간 연산을 Depthwise 3×3과 Pointwise 1×1로 나눕니다. 입력은 32×32 RGB 영상이고, 분류기는 CIFAR-10 클래스별 Logit 10개를 출력합니다.

학습 방법에 따라 달라지는 것은 **Student가 정답 외에 함께 받는 학습 신호**입니다.

**코드 셀 3 · 경량 ResNet-18 정의**

BasicBlock의 첫 3×3 연산을 Depthwise와 Pointwise로 나눈 경량 ResNet-18 모델의 생성 함수를 정의합니다.

In [ ]:
# 기존 BasicBlock의 3×3 연산을 Depthwise와 Pointwise로 나누어 경량 모델을 만든다.
def build_light_resnet18():
    __all__ = ['ResNet', 'resnet18']

    def conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
        """3x3 convolution with padding"""
        return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                         padding=dilation, groups=in_planes, bias=False, dilation=dilation)

    def conv1x1(in_planes, out_planes, stride=1):
        """1x1 convolution"""
        return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)

    # 주 경로에서 특징을 계산하고 Residual 입력을 더한다. 크기가 바뀌는 블록에서는 Residual도 변환한다.
    class BasicBlock(nn.Module):
        expansion = 1

        def __init__(self, inplanes, planes, stride=1, downsample=None, groups=1,
                     base_width=64, dilation=1, norm_layer=None):
            super(BasicBlock, self).__init__()

            if norm_layer is None:
                norm_layer = nn.BatchNorm2d
            if groups != 1 or base_width != 64:
                raise ValueError(
                    'BasicBlock only supports groups=1 and base_width=64')
            if dilation > 1:
                raise NotImplementedError(
                    "Dilation > 1 not supported in BasicBlock")
            # Both self.conv1 and self.downsample layers downsample the input when stride != 1
            # 3×3 Depthwise에서 채널별 공간 특징을 계산하고, 1×1 Pointwise에서 채널을 조합한다.
            self.dwconv33_1 = conv3x3(inplanes, planes, stride)
            self.conv11_1 = conv1x1(planes, planes)
            self.bn1 = norm_layer(planes)
            self.relu = nn.ReLU(inplace=True)

            self.dwconv33_2 = conv3x3(planes, planes)
            self.conv11_2 = conv1x1(planes, planes)
            self.bn2 = norm_layer(planes)
            self.downsample = downsample
            self.stride = stride

        def forward(self, x):
            identity = x

            out = self.dwconv33_1(x)
            out = self.conv11_1(out)
            out = self.bn1(out)
            out = self.relu(out)

            out = self.dwconv33_2(out)
            out = self.conv11_2(out)
            out = self.bn2(out)

            if self.downsample is not None:
                identity = self.downsample(x)

            out += identity
            out = self.relu(out)

            return out

    # 입력 stem과 Residual stage를 연결하고, Global Average Pooling 뒤에서 클래스별
    # Logit을 계산한다.
    class ResNet(nn.Module):

        def __init__(self, block, layers, num_classes=1000, zero_init_residual=False,
                     groups=1, width_per_group=64, replace_stride_with_dilation=None,
                     norm_layer=None):
            super(ResNet, self).__init__()
            if norm_layer is None:
                norm_layer = nn.BatchNorm2d
            self._norm_layer = norm_layer

            self.inplanes = 64
            self.dilation = 1
            if replace_stride_with_dilation is None:
                # each element in the tuple indicates if we should replace
                # the 2x2 stride with a dilated convolution instead
                replace_stride_with_dilation = [False, False, False]
            if len(replace_stride_with_dilation) != 3:
                raise ValueError("replace_stride_with_dilation should be None "
                                 "or a 3-element tuple, got {}".format(replace_stride_with_dilation))
            self.groups = groups
            self.base_width = width_per_group
            self.conv1 = nn.Conv2d(3, self.inplanes, kernel_size=7, stride=2, padding=3,
                                   bias=False)

            self.bn1 = norm_layer(self.inplanes)
            self.relu = nn.ReLU(inplace=True)
            self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
            # 네 stage는 64·128·256·512채널을 사용한다. 첫 블록의 stride로 해상도를 조정한다.
            self.layer1 = self._make_layer(block, 64, layers[0])
            self.layer2 = self._make_layer(block, 128, layers[1], stride=2,
                                           dilate=replace_stride_with_dilation[0])
            self.layer3 = self._make_layer(block, 256, layers[2], stride=2,
                                           dilate=replace_stride_with_dilation[1])
            self.layer4 = self._make_layer(block, 512, layers[3], stride=2,
                                           dilate=replace_stride_with_dilation[2])
            self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
            self.fc = nn.Linear(512 * block.expansion, num_classes)

            # Convolution과 정규화 계층의 학습 초기값을 설정한다.
            for m in self.modules():
                if isinstance(m, nn.Conv2d):
                    nn.init.kaiming_normal_(
                        m.weight, mode='fan_out', nonlinearity='relu')
                elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                    nn.init.constant_(m.weight, 1)
                    nn.init.constant_(m.bias, 0)

            # Zero-initialize the last BN in each residual branch,
            # so that the residual branch starts with zeros, and each residual block behaves like an identity.
            # This improves the model by 0.2~0.3% according to https://arxiv.org/abs/1706.02677
            if zero_init_residual:
                for m in self.modules():
                    if isinstance(m, Bottleneck):
                        nn.init.constant_(m.bn3.weight, 0)
                    elif isinstance(m, BasicBlock):
                        nn.init.constant_(m.bn2.weight, 0)

        # 첫 블록에서 필요한 Residual 변환을 적용하고, 뒤의 블록은 같은 출력 크기로 이어 붙인다.
        def _make_layer(self, block, planes, blocks, stride=1, dilate=False):
            norm_layer = self._norm_layer
            downsample = None
            previous_dilation = self.dilation
            if dilate:
                self.dilation *= stride
                stride = 1
            if stride != 1 or self.inplanes != planes * block.expansion:
                downsample = nn.Sequential(
                    conv1x1(self.inplanes, planes * block.expansion, stride),
                    norm_layer(planes * block.expansion),
                )

            layers = []
            layers.append(block(self.inplanes, planes, stride, downsample, self.groups,
                                self.base_width, previous_dilation, norm_layer))
            self.inplanes = planes * block.expansion
            for _ in range(1, blocks):
                layers.append(block(self.inplanes, planes, groups=self.groups,
                                    base_width=self.base_width, dilation=self.dilation,
                                    norm_layer=norm_layer))

            return nn.Sequential(*layers)

        # 입력에서 stem, 네 stage, Pooling, 최종 Linear Layer로 이어지는 순전파다.
        def _forward_impl(self, x):
            # See note [TorchScript super()]
            x = self.conv1(x)
            x = self.bn1(x)
            x = self.relu(x)
            x = self.maxpool(x)

            x = self.layer1(x)
            x = self.layer2(x)
            x = self.layer3(x)
            x = self.layer4(x)

            x = self.avgpool(x)
            x = torch.flatten(x, 1)
            x = self.fc(x)

            return x

        def forward(self, x):
            return self._forward_impl(x)

    # CIFAR-10에 맞게 첫 Conv2d를 3×3, stride 1로 설정하고 출력 클래스 수를 10으로 둔다.
    def resnet_18_cifar_lightweight():
        r"""
            Modify the resnet 18 network in order to run on cifar-10 dataset

            To enhance the accuracy, the (kernel_size, stride, padding) of conv1 is modified to (3, 1, 1)
            referenced by <https://github.com/akamaster/pytorch_resnet_cifar10>
        """
        model = ResNet(block=BasicBlock, layers=[2, 2, 2, 2], num_classes=10)

        model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1,
                                bias=False)

        return model

    return resnet_18_cifar_lightweight()

**코드 셀 4 · 분류 정확도 평가 함수**

평가 모드에서 각 이미지의 예측 클래스와 정답을 비교해 정확도를 백분율로 반환하는 함수를 정의합니다.

In [ ]:
# 평가 모드에서 각 영상의 최댓값 클래스를 정답과 비교해 전체 정확도를 집계한다.
@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        correct += (model(images).argmax(1) == labels).sum().item()
        total += labels.numel()
    return 100 * correct / total

## 3. 고정 Teacher 준비

[CIFAR-10 ResNet-18 공개 체크포인트](https://huggingface.co/edadaltocg/resnet18_cifar10)를 다운로드하거나 기존 캐시에서 불러옵니다. Teacher를 평가 모드에 두고 파라미터의 gradient를 끕니다. Student를 학습하는 동안 Teacher의 출력은 고정된 학습 목표로 사용됩니다.

표시되는 **Teacher 정확도**는 체크포인트와 입력 전처리를 확인하는 기준입니다.

**코드 셀 5 · 사전 학습 Teacher 불러오기**

CIFAR-10으로 사전 학습한 ResNet-18을 불러와 평가 모드에서 가중치를 고정하고 테스트 정확도를 출력합니다.

In [ ]:
from torchvision.models import resnet18

# CIFAR-10으로 사전 학습한 공개 Teacher의 체크포인트를 불러온다.
TEACHER_URL = ("https://huggingface.co/edadaltocg/resnet18_cifar10/resolve/"
               "ccacbae365101b9bfa45e1f17f4acac30c2601b2/pytorch_model.bin")
teacher = resnet18(weights=None, num_classes=10)
teacher.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
teacher.maxpool = nn.Identity()
state_dict = torch.hub.load_state_dict_from_url(
    TEACHER_URL, model_dir=str(ASSET_ROOT), file_name="teacher.bin",
    map_location="cpu", weights_only=True
)
teacher.load_state_dict(state_dict, strict=True)
# Offline KD에서는 Teacher를 평가 모드에 두고 가중치를 고정한다.
teacher = teacher.to(DEVICE).eval().requires_grad_(False)
print(f"Teacher 정확도: {evaluate(teacher, test_loader):.2f}%")

## 4. Temperature와 Soft Target

**Temperature T**는 Softmax 직전에 모든 클래스의 Logit을 나누는 값입니다. 같은 영상의 Logit으로 **T=1과 T=4**를 비교합니다. 출력에서 클래스 순위와 확률의 차이가 어떻게 달라지는지 확인합니다.

`response_kd()`는 Student 로그 확률과 Teacher 확률의 KL divergence를 계산합니다. 두 모델에 같은 T를 적용하고 손실에 T²를 곱합니다.

**코드 셀 6 · Temperature와 손실 가중치 설정**

Soft Target에 사용할 Temperature를 4.0으로, Response KD Loss에 곱할 가중치를 0.25로 설정합니다.

In [ ]:
# 모든 Logit을 같은 양수 T로 나눈 뒤 Softmax를 적용한다. 함께 둔 계수는 KD 손실의 비중을 정한다.
TEMPERATURE = 4.0
RESPONSE_WEIGHT = 0.25

**코드 셀 7 · Response KD Loss 정의**

Temperature를 적용한 Student와 목표 모델의 확률 분포 사이 KL divergence에 Temperature의 제곱을 곱해 손실을 계산합니다.

In [ ]:
# Student 로그 확률과 고정된 목표 확률의 KL divergence에 T²를 곱해 KD Loss를 계산한다.
def response_kd(student_logits, teacher_logits, temperature):
    log_student = F.log_softmax(student_logits / temperature, dim=1)
    soft_target = F.softmax(teacher_logits.detach() / temperature, dim=1)
    return F.kl_div(log_student, soft_target, reduction="batchmean") * temperature**2

**코드 셀 8 · Temperature에 따른 Teacher 예측 비교**

같은 테스트 이미지 한 장에서 Temperature 1과 설정값을 적용했을 때 Teacher의 상위 세 클래스와 확률을 출력합니다.

In [ ]:
views, _ = next(iter(test_loader))
with torch.inference_mode():
    logits = teacher(views[:1].to(DEVICE))
    # 같은 영상의 Logit을 사용해 T만 바꿨을 때 상위 확률이 어떻게 달라지는지 비교한다.
    for temperature in (1.0, TEMPERATURE):
        values, classes = (logits / temperature).softmax(dim=1).topk(3)
        top = [(test_data.classes[k], round(v, 4))
               for k, v in zip(classes[0].tolist(), values[0].tolist())]
        print(f"T={temperature:g}: {top}")

## 5. Student 학습과 평가

정답 레이블의 **Cross-Entropy**와 Teacher 분포에 대한 **KD Loss**를 계산합니다. 코드에서는 KD Loss에 `RESPONSE_WEIGHT`를 곱해 정답 손실과 더한 뒤 Student의 Weight를 갱신합니다.

마지막 출력에서 **배치 수, 시간, 정답 손실, KD Loss, 테스트 정확도**를 확인합니다. 두 손실은 계산 방식과 크기 단위가 다르므로, 손실값의 절댓값만으로 어느 쪽의 영향이 더 크다고 판단하기는 어렵습니다.

**코드 셀 9 · Response-Based KD 학습과 평가**

정답과 Teacher의 출력 확률을 함께 사용해 Student를 학습하고 마지막 배치의 두 손실과 테스트 정확도를 출력합니다.

In [ ]:
# 독립적인 경량 Student를 준비해 학습 가능한 파라미터만 갱신한다.
student = build_light_resnet18().to(DEVICE)
optimizer = torch.optim.SGD(student.parameters(), lr=0.03, momentum=0.9)
start = time.perf_counter()
history = []
for images, labels in islice(train_loader, TRAIN_STEPS):
    images, labels = images.to(DEVICE), labels.to(DEVICE)
    # Teacher는 학습 목표를 만드는 데 사용한다. 이 Forward에서는 Teacher의 gradient를 계산하지
    # 않는다.
    with torch.no_grad():
        teacher_logits = teacher(images)

    optimizer.zero_grad(set_to_none=True)
    logits = student(images)
    ce = F.cross_entropy(logits, labels)
    kd = response_kd(logits, teacher_logits, TEMPERATURE)
    # 정답 레이블과 Teacher 출력 확률의 두 손실을 합쳐 Student에 역전파한다.
    (ce + RESPONSE_WEIGHT * kd).backward()
    optimizer.step()
    history.append((ce.item(), kd.item()))

print(f"{len(history)} batches · {time.perf_counter() - start:.1f} s")
print(f"마지막 배치: CE {history[-1][0]:.3f} | Response KD {history[-1][1]:.3f}")
print(f"Student 정확도: {evaluate(student, test_loader):.2f}%")